---
# ═══════════════════════════════════════
# 🔧 PHASE 2 — PRÉPARATION & PRÉTRAITEMENT
# ═══════════════════════════════════════

## 2.1 — Importation des bibliothèques de prétraitement

> On importe les outils Scikit-learn pour l'encodage, la normalisation et la séparation des données.

In [ ]:
# ── Bibliothèques Scikit-learn pour le prétraitement
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import copy

print('✅ Bibliothèques de prétraitement importées')

## 2.2 — Copie de travail du DataFrame

> On travaille sur une copie pour ne jamais altérer le dataset original.  
> C'est une bonne pratique essentielle en Data Science.

In [ ]:
# ── Création d'une copie propre pour le prétraitement
df_prep = df.copy()

print(f'✅ Copie créée : df_prep — {df_prep.shape[0]:,} lignes × {df_prep.shape[1]} colonnes')
print(f'   Dataset original "df" préservé intact ({df.shape})')

## 2.3 — Conversion de la date et features temporelles

> On convertit la colonne `date` (type string) en `datetime`.  
> On extrait ensuite des features temporelles supplémentaires utiles pour les modèles.

In [ ]:
# ── Conversion de la colonne date en datetime
df_prep['date'] = pd.to_datetime(df_prep['date'])

print('── Avant/Après conversion ──')
print(f'   Type avant : str  →  Type après : {df_prep["date"].dtype}')

# ── Extraction de features temporelles enrichies
df_prep['annee']        = df_prep['date'].dt.year
df_prep['mois']         = df_prep['date'].dt.month
df_prep['jour']         = df_prep['date'].dt.day
df_prep['jour_semaine'] = df_prep['date'].dt.dayofweek   # 0=Lundi, 6=Dimanche
df_prep['trimestre']    = df_prep['date'].dt.quarter
df_prep['semaine_annee']= df_prep['date'].dt.isocalendar().week.astype(int)
df_prep['is_weekend']   = df_prep['jour_semaine'].isin([5, 6]).astype(int)
df_prep['is_mois_debut']= (df_prep['jour'] <= 10).astype(int)  # début de mois
df_prep['is_mois_fin']  = (df_prep['jour'] >= 21).astype(int)  # fin de mois

nouvelles_cols = ['annee','mois','jour','jour_semaine','trimestre',
                  'semaine_annee','is_weekend','is_mois_debut','is_mois_fin']
print()
print('── Features temporelles créées ──')
print(df_prep[['date'] + nouvelles_cols].head(3))

## 2.4 — Vérification et traitement des valeurs manquantes

> Même si le dataset est propre, on implémente le code de traitement pour montrer la démarche rigoureuse.  
> En production, des valeurs manquantes peuvent apparaître dans les nouvelles données.

In [ ]:
# ── Vérification finale des valeurs manquantes après manipulations
missing_final = df_prep.isnull().sum()
print('── Vérification valeurs manquantes (après ajout features) ──')
print(missing_final[missing_final > 0] if missing_final.sum() > 0
      else '✅ Aucune valeur manquante — Rien à traiter')

# ── Code de traitement (à activer si données futures ont des NaN)
print()
print('── Code de traitement (prêt pour production) ──')

# Colonnes numériques → remplissage par médiane (robuste aux outliers)
num_cols_fill = df_prep.select_dtypes(include=[np.number]).columns.tolist()
df_prep[num_cols_fill] = df_prep[num_cols_fill].fillna(df_prep[num_cols_fill].median())

# Colonnes catégorielles → remplissage par mode (valeur la plus fréquente)
cat_cols_fill = df_prep.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols_fill:
    df_prep[col].fillna(df_prep[col].mode()[0], inplace=True)

print(f'   → Colonnes numériques  ({len(num_cols_fill)}) : remplissage par médiane')
print(f'   → Colonnes catégorielles ({len(cat_cols_fill)}) : remplissage par mode')
print(f'   → Valeurs manquantes restantes : {df_prep.isnull().sum().sum()}')

## 2.5 — Suppression des doublons

> On supprime les doublons sur la clé logique du dataset : date + article + dépôt.

In [ ]:
# ── Suppression des doublons
avant = len(df_prep)
df_prep.drop_duplicates(inplace=True)
apres = len(df_prep)

print(f'── Suppression des doublons ──')
print(f'   Lignes avant : {avant:,}')
print(f'   Lignes après : {apres:,}')
print(f'   Doublons supprimés : {avant - apres}')
print(f'\n✅ Dataset : {apres:,} lignes × {df_prep.shape[1]} colonnes')

## 2.6 — Feature Engineering : Nouvelles Variables

> On crée de nouvelles variables à partir des variables existantes.  
> Ces features enrichies améliorent significativement les performances des modèles ML.

In [ ]:
# ── Feature Engineering : variables dérivées

# 1. Marge brute et marge en pourcentage
df_prep['marge_brute']     = df_prep['prix_vente'] - df_prep['prix_achat']
df_prep['marge_pct']       = (df_prep['marge_brute'] / df_prep['prix_achat'] * 100).round(2)

# 2. Chiffre d'affaires journalier
df_prep['chiffre_affaires']= df_prep['prix_vente'] * df_prep['vente_qte']

# 3. Coût d'achat journalier
df_prep['cout_achat']      = df_prep['prix_achat'] * df_prep['achat_qte']

# 4. Ratio stock : niveau de remplissage relatif
df_prep['ratio_stock']     = (df_prep['stock_final'] / (df_prep['stock_final'] + df_prep['vente_qte'] + 0.001)).round(4)

# 5. Pression de la demande : ventes rapportées au stock initial
df_prep['pression_demande']= (df_prep['vente_qte'] / (df_prep['stock_initial'] + 0.001)).round(4)

# 6. Écart relatif entre movennes court terme et long terme
df_prep['ecart_tendance']  = (df_prep['moyenne_vente_7j'] - df_prep['moyenne_vente_30j']).round(3)

# 7. Accélération des ventes (tendance court terme par rapport au long terme)
df_prep['acceleration_vente'] = (df_prep['moyenne_vente_7j'] / (df_prep['moyenne_vente_30j'] + 0.001)).round(3)

nouvelles_features = [
    ('marge_brute',      'Prix vente − Prix achat'),
    ('marge_pct',        'Marge en % du prix achat'),
    ('chiffre_affaires', 'Prix vente × Qté vendue'),
    ('cout_achat',       'Prix achat × Qté achetée'),
    ('ratio_stock',      'Stock final / (Stock + Ventes)'),
    ('pression_demande', 'Ventes / Stock initial'),
    ('ecart_tendance',   'Moy 7j − Moy 30j'),
    ('acceleration_vente','Moy 7j / Moy 30j'),
]

print('══ Nouvelles features créées ══')
for feat, desc in nouvelles_features:
    print(f'   ✅ {feat:<25} → {desc}')

print(f'\n   Colonnes totales : {df_prep.shape[1]}')

In [ ]:
# ── Lag Features : valeurs décalées dans le temps
# Important pour les modèles de séries temporelles
# On trie d'abord par article, dépôt et date

df_prep = df_prep.sort_values(['article_id', 'depot_id', 'date']).reset_index(drop=True)

# Lag 1, 3, 7 jours sur les ventes (par groupe article+dépôt)
for lag in [1, 3, 7]:
    df_prep[f'vente_lag_{lag}'] = (
        df_prep.groupby(['article_id', 'depot_id'])['vente_qte']
               .shift(lag)
    )

# Lag 1 jour sur le stock final
df_prep['stock_lag_1'] = (
    df_prep.groupby(['article_id', 'depot_id'])['stock_final']
           .shift(1)
)

print('── Lag Features créées ──')
print('   ✅ vente_lag_1  → Ventes du jour J-1')
print('   ✅ vente_lag_3  → Ventes du jour J-3')
print('   ✅ vente_lag_7  → Ventes du jour J-7')
print('   ✅ stock_lag_1  → Stock final du jour J-1')

# Les lags créent des NaN sur les premières lignes de chaque groupe
n_nan_lag = df_prep['vente_lag_7'].isnull().sum()
print(f'\n   ⚠️  Valeurs NaN créées par les lags : {n_nan_lag:,}')
print(f'   → Ces NaN seront supprimés ci-dessous')

## 2.7 — Gestion des NaN issus des Lag Features

> Les features de décalage (lags) créent des NaN pour les premières observations de chaque article.  
> On supprime ces lignes car on ne peut pas les compléter sans inventer des données.

In [ ]:
# ── Suppression des NaN issus des lags
avant_lag = len(df_prep)
df_prep.dropna(subset=['vente_lag_1', 'vente_lag_3', 'vente_lag_7', 'stock_lag_1'], inplace=True)
df_prep = df_prep.reset_index(drop=True)
apres_lag = len(df_prep)

print(f'── Suppression NaN (lags) ──')
print(f'   Lignes avant : {avant_lag:,}')
print(f'   Lignes après : {apres_lag:,}')
print(f'   Lignes perdues : {avant_lag - apres_lag:,} ({(avant_lag - apres_lag)/avant_lag*100:.1f}%)')
print(f'\n✅ Vérification finale : {df_prep.isnull().sum().sum()} valeur(s) manquante(s)')

## 2.8 — Encodage des Variables Catégorielles

> Les algorithmes ML ne peuvent pas traiter des chaînes de caractères.  
> On convertit les variables catégorielles en valeurs numériques avec LabelEncoder.  
> On conserve les encodeurs pour les utiliser lors des prédictions futures.

In [ ]:
# ── Encodage LabelEncoder des variables catégorielles
# On conserve le DataFrame original ET on crée une version encodée

df_encoded = df_prep.copy()

# Variables à encoder
cat_to_encode = ['categorie', 'matiere', 'ville', 'article_id', 'depot_id']

# Dictionnaire pour stocker les encodeurs (pour le décodage futur)
encoders = {}

print('── Encodage LabelEncoder ──')
for col in cat_to_encode:
    le = LabelEncoder()
    df_encoded[col + '_enc'] = le.fit_transform(df_encoded[col])
    encoders[col] = le
    print(f'   ✅ {col:<15} → {col}_enc  | Classes : {list(le.classes_)}')

print()
print('── Correspondances d\'encodage ──')
for col in ['categorie', 'matiere', 'ville']:
    le = encoders[col]
    mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
    print(f'   {col} : {mapping}')

In [ ]:
# ── Vérification de l'encodage
print('── Vérification encodage (5 premières lignes) ──')
verification_cols = ['categorie', 'categorie_enc', 'matiere', 'matiere_enc', 'ville', 'ville_enc']
display(df_encoded[verification_cols].head(10))

## 2.9 — Sélection et Organisation des Features

> On définit les jeux de features pour chacune des 3 tâches de modélisation.  
> C'est l'étape clé qui détermine ce qu'on donne au modèle comme entrées et comme sorties.

In [ ]:
# ── Définition des features communes à toutes les tâches
features_communes = [
    # Identifiants encodés
    'categorie_enc', 'matiere_enc', 'ville_enc', 'article_id_enc', 'depot_id_enc',
    # Stock
    'stock_initial', 'stock_final', 'stock_lag_1', 'achat_qte',
    # Prix
    'prix_achat', 'prix_vente', 'marge_brute', 'marge_pct',
    # Indicateurs stock
    'rotation_stock', 'couverture_stock', 'ecart_stock',
    # Moyennes ventes
    'moyenne_vente_7j', 'moyenne_vente_30j',
    # Lag features
    'vente_lag_1', 'vente_lag_3', 'vente_lag_7',
    # Features temporelles
    'annee', 'mois', 'jour', 'jour_semaine', 'trimestre', 'semaine_annee',
    'is_weekend', 'is_mois_debut', 'is_mois_fin',
    # Features dérivées
    'ratio_stock', 'pression_demande', 'ecart_tendance', 'acceleration_vente'
]

# ── TÂCHE A : Prévision des ventes
features_A = features_communes.copy()
target_A   = 'vente_qte'

# ── TÂCHE B : Détection rupture/surstock
features_B = features_communes + ['vente_qte', 'couverture_stock', 'rotation_stock']
# On crée une variable cible combinée : 0=Normal, 1=Surstock, 2=Rupture
df_encoded['alerte_stock'] = 0
df_encoded.loc[df_encoded['surstock']      == 1, 'alerte_stock'] = 1
df_encoded.loc[df_encoded['rupture_stock'] == 1, 'alerte_stock'] = 2
target_B = 'alerte_stock'

# ── TÂCHE C : Recommandation réapprovisionnement
features_C = features_communes + ['vente_qte', 'couverture_stock']
target_C   = 'achat_qte'

print('══ Features par Tâche ML ══')
print(f'   Tâche A — Prévision ventes      : {len(features_A)} features → cible : {target_A}')
print(f'   Tâche B — Détection alertes     : {len(features_B)} features → cible : {target_B}')
print(f'   Tâche C — Recommandation réapro : {len(features_C)} features → cible : {target_C}')
print()
print(f'   Distribution de alerte_stock :')
dist_alerte = df_encoded['alerte_stock'].value_counts()
labels_alerte = {0: 'Normal', 1: 'Surstock', 2: 'Rupture'}
for k, v in dist_alerte.items():
    print(f'     {labels_alerte[k]:<10} : {v:,} ({v/len(df_encoded)*100:.2f}%)')

## 2.10 — Normalisation / Standardisation

> La standardisation centre les données autour de 0 avec un écart-type de 1.  
> C'est nécessaire pour les modèles sensibles à l'échelle des données (Régression logistique, etc.).
> XGBoost et Random Forest n'en ont pas besoin, mais on l'applique pour avoir un pipeline uniforme.

In [ ]:
# ── Standardisation avec StandardScaler
# On standardise uniquement les colonnes numériques continues

cols_to_scale = [
    'stock_initial', 'stock_final', 'stock_lag_1', 'achat_qte',
    'prix_achat', 'prix_vente', 'marge_brute',
    'rotation_stock', 'couverture_stock',
    'moyenne_vente_7j', 'moyenne_vente_30j',
    'vente_lag_1', 'vente_lag_3', 'vente_lag_7',
    'ratio_stock', 'pression_demande', 'ecart_tendance', 'acceleration_vente'
]

# Filtrer uniquement les colonnes existantes
cols_to_scale = [c for c in cols_to_scale if c in df_encoded.columns]

scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])

print('── StandardScaler appliqué ──')
print(f'   Colonnes standardisées : {len(cols_to_scale)}')
print()

# Vérification : moyenne ≈ 0, std ≈ 1
verification_scale = pd.DataFrame({
    'Moyenne avant' : df_encoded[cols_to_scale[:5]].mean().round(3).values,
    'Std avant'     : df_encoded[cols_to_scale[:5]].std().round(3).values,
    'Moyenne après' : df_scaled[cols_to_scale[:5]].mean().round(4).values,
    'Std après'     : df_scaled[cols_to_scale[:5]].std().round(4).values,
}, index=cols_to_scale[:5])

print('── Vérification (5 premières colonnes) ──')
display(verification_scale)
print('\n✅ Standardisation correcte : Moyenne ≈ 0, Std ≈ 1')

In [ ]:
# ── Visualisation avant / après standardisation
cols_vis = ['prix_vente', 'couverture_stock', 'rotation_stock', 'vente_lag_7']

fig, axes = plt.subplots(2, len(cols_vis), figsize=(20, 8))

for j, col in enumerate(cols_vis):
    # Avant
    axes[0, j].hist(df_encoded[col], bins=40, color='#e74c3c', alpha=0.7, edgecolor='white')
    axes[0, j].set_title(f'{col}\n(Avant normalisation)', fontsize=10, fontweight='bold')
    axes[0, j].set_ylabel('Fréquence')

    # Après
    axes[1, j].hist(df_scaled[col], bins=40, color='#2ecc71', alpha=0.7, edgecolor='white')
    axes[1, j].set_title(f'{col}\n(Après standardisation)', fontsize=10, fontweight='bold')
    axes[1, j].set_ylabel('Fréquence')

plt.suptitle('Effet de la Standardisation (StandardScaler)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('phase2_standardisation.png', bbox_inches='tight', dpi=150)
plt.show()

## 2.11 — Séparation Train / Validation / Test

> On découpe le dataset en 3 parties :  
> - **Train** (70%) : pour entraîner les modèles  
> - **Validation** (15%) : pour régler les hyperparamètres  
> - **Test** (15%) : pour l'évaluation finale (données jamais vues)
>
> ⚠️ Pour les séries temporelles, on **respecte l'ordre chronologique** (pas de mélange aléatoire).

In [ ]:
# ── Séparation chronologique (respect de l'ordre temporel)

# On trie par date pour respecter l'ordre temporel
df_model = df_scaled.sort_values('date').reset_index(drop=True)

# Calcul des indices de coupure
n = len(df_model)
idx_train = int(n * 0.70)
idx_val   = int(n * 0.85)

# ─────────────────────────────────────────────────────
# TÂCHE A — Prévision des ventes
# ─────────────────────────────────────────────────────
X_A = df_model[[c for c in features_A if c in df_model.columns]]
y_A = df_model[target_A]

X_A_train = X_A.iloc[:idx_train]
X_A_val   = X_A.iloc[idx_train:idx_val]
X_A_test  = X_A.iloc[idx_val:]

y_A_train = y_A.iloc[:idx_train]
y_A_val   = y_A.iloc[idx_train:idx_val]
y_A_test  = y_A.iloc[idx_val:]

# ─────────────────────────────────────────────────────
# TÂCHE B — Détection rupture/surstock
# ─────────────────────────────────────────────────────
X_B = df_model[[c for c in features_B if c in df_model.columns]]
y_B = df_model[target_B]

X_B_train = X_B.iloc[:idx_train]
X_B_val   = X_B.iloc[idx_train:idx_val]
X_B_test  = X_B.iloc[idx_val:]

y_B_train = y_B.iloc[:idx_train]
y_B_val   = y_B.iloc[idx_train:idx_val]
y_B_test  = y_B.iloc[idx_val:]

# ─────────────────────────────────────────────────────
# TÂCHE C — Recommandation réapprovisionnement
# ─────────────────────────────────────────────────────
X_C = df_model[[c for c in features_C if c in df_model.columns]]
y_C = df_model[target_C]

X_C_train = X_C.iloc[:idx_train]
X_C_val   = X_C.iloc[idx_train:idx_val]
X_C_test  = X_C.iloc[idx_val:]

y_C_train = y_C.iloc[:idx_train]
y_C_val   = y_C.iloc[idx_train:idx_val]
y_C_test  = y_C.iloc[idx_val:]

# ─────────────────────────────────────────────────────
# Affichage du résumé
# ─────────────────────────────────────────────────────
print('══ Découpage Train / Validation / Test ══')
print(f'   Méthode         : Séparation chronologique (respect ordre temporel)')
print(f'   Total lignes    : {n:,}')
print()
print(f'   🟦 Train        : {len(X_A_train):,} lignes ({len(X_A_train)/n*100:.0f}%) '
      f'| du {df_model["date"].iloc[0].date()} au {df_model["date"].iloc[idx_train-1].date()}')
print(f'   🟨 Validation   : {len(X_A_val):,} lignes ({len(X_A_val)/n*100:.0f}%)  '
      f'| du {df_model["date"].iloc[idx_train].date()} au {df_model["date"].iloc[idx_val-1].date()}')
print(f'   🟥 Test         : {len(X_A_test):,} lignes ({len(X_A_test)/n*100:.0f}%)  '
      f'| du {df_model["date"].iloc[idx_val].date()} au {df_model["date"].iloc[-1].date()}')

In [ ]:
# ── Visualisation de la répartition train/val/test sur la timeline
fig, ax = plt.subplots(figsize=(16, 4))

# Ventes journalières moyennes
ventes_jour = df_model.groupby('date')['vente_qte'].mean()

ax.plot(df_model['date'], df_model['vente_qte'], alpha=0.15, color='gray', linewidth=0.5)

# Colorisation des zones
date_train_end = df_model['date'].iloc[idx_train - 1]
date_val_end   = df_model['date'].iloc[idx_val - 1]
date_test_end  = df_model['date'].iloc[-1]
date_start     = df_model['date'].iloc[0]

ax.axvspan(date_start, date_train_end,  alpha=0.25, color='blue',   label=f'Train (70%) — {len(X_A_train):,} lignes')
ax.axvspan(date_train_end, date_val_end, alpha=0.25, color='orange', label=f'Validation (15%) — {len(X_A_val):,} lignes')
ax.axvspan(date_val_end, date_test_end,  alpha=0.25, color='red',    label=f'Test (15%) — {len(X_A_test):,} lignes')

ax.axvline(date_train_end, color='blue',   linestyle='--', linewidth=1.5)
ax.axvline(date_val_end,   color='orange', linestyle='--', linewidth=1.5)

ax.set_title('Découpage Chronologique — Train / Validation / Test', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Quantité vendue (vente_qte)')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase2_train_val_test_split.png', bbox_inches='tight', dpi=150)
plt.show()

## 2.12 — Sauvegarde des données préparées

> On sauvegarde les données prétraitées pour pouvoir les réutiliser directement dans la Phase 3 (Modélisation).

In [ ]:
# ── Sauvegarde du DataFrame préparé complet
df_encoded.to_csv('dataset_prepared.csv', index=False)
df_scaled.to_csv('dataset_scaled.csv', index=False)

# ── Sauvegarde des jeux de données par tâche
import joblib

# Sauvegarde des splits pour chaque tâche
joblib.dump({
    'X_train': X_A_train, 'X_val': X_A_val, 'X_test': X_A_test,
    'y_train': y_A_train, 'y_val': y_A_val, 'y_test': y_A_test,
    'features': [c for c in features_A if c in df_model.columns]
}, 'data_tache_A.pkl')

joblib.dump({
    'X_train': X_B_train, 'X_val': X_B_val, 'X_test': X_B_test,
    'y_train': y_B_train, 'y_val': y_B_val, 'y_test': y_B_test,
    'features': [c for c in features_B if c in df_model.columns]
}, 'data_tache_B.pkl')

joblib.dump({
    'X_train': X_C_train, 'X_val': X_C_val, 'X_test': X_C_test,
    'y_train': y_C_train, 'y_val': y_C_val, 'y_test': y_C_test,
    'features': [c for c in features_C if c in df_model.columns]
}, 'data_tache_C.pkl')

# Sauvegarde du scaler et des encodeurs
joblib.dump(scaler,   'scaler.pkl')
joblib.dump(encoders, 'encoders.pkl')

print('✅ Fichiers sauvegardés :')
print('   📄 dataset_prepared.csv  → DataFrame encodé (sans standardisation)')
print('   📄 dataset_scaled.csv    → DataFrame encodé + standardisé')
print('   📦 data_tache_A.pkl      → Splits Train/Val/Test — Tâche A (ventes)')
print('   📦 data_tache_B.pkl      → Splits Train/Val/Test — Tâche B (alertes)')
print('   📦 data_tache_C.pkl      → Splits Train/Val/Test — Tâche C (réapro)')
print('   🔧 scaler.pkl            → StandardScaler entraîné')
print('   🔧 encoders.pkl          → Dictionnaire LabelEncoders')

## 2.13 — Synthèse de la Phase 2

In [ ]:
print('=' * 70)
print('  🔧 SYNTHÈSE — PHASE 2 : PRÉPARATION & PRÉTRAITEMENT')
print('=' * 70)
print()
print('  ✅ ÉTAPES RÉALISÉES')
print('     1. Copie de sécurité du dataset original')
print('     2. Conversion date (str → datetime)')
print('     3. Vérification valeurs manquantes → 0 valeur manquante')
print('     4. Vérification et suppression doublons → 0 doublon')
print('     5. Feature Engineering : 8 nouvelles variables créées')
print('     6. Lag Features : vente_lag_1, vente_lag_3, vente_lag_7, stock_lag_1')
print('     7. Suppression NaN issus des lags')
print('     8. Encodage LabelEncoder : categorie, matiere, ville, article_id, depot_id')
print('     9. Standardisation StandardScaler : 18 colonnes numériques')
print('    10. Séparation chronologique 70/15/15 (Train/Val/Test)')
print('    11. Sauvegarde des fichiers pour la Phase 3')
print()
print('  📊 STATISTIQUES FINALES')
print(f'     Dataset final     : {len(df_scaled):,} lignes × {df_scaled.shape[1]} colonnes')
print(f'     Features Tâche A  : {len([c for c in features_A if c in df_model.columns])} features')
print(f'     Features Tâche B  : {len([c for c in features_B if c in df_model.columns])} features')
print(f'     Features Tâche C  : {len([c for c in features_C if c in df_model.columns])} features')
print(f'     Train             : {len(X_A_train):,} lignes')
print(f'     Validation        : {len(X_A_val):,} lignes')
print(f'     Test              : {len(X_A_test):,} lignes')
print()
print('  ➡️  PROCHAINE ÉTAPE : Phase 3 — Modélisation ML')
print('     → Tâche A : Random Forest vs XGBoost (prévision ventes)')
print('     → Tâche B : Logistic Regression vs Random Forest (alertes stock)')
print('     → Tâche C : Linear Regression vs Gradient Boosting (réapprovisionnement)')
print('=' * 70)